# Experiment: Colab A100 - 7B 풀파워 세팅

## 실험 설정 요약

| 항목 | 설정 | 이유 |
|------|------|------|
| **모델** | Qwen2.5-VL-7B-Instruct | 3B 대비 파라미터 2배 -> 이미지 이해력/추론력 향상. A100 40GB면 4bit로 여유 있게 돌림 |
| **양자화** | 4bit (NF4) | A100에서도 4bit 쓰면 VRAM 여유 -> 해상도와 배치를 올릴 수 있음 |
| **데이터** | 전체 train + dev pseudo-label | 200개 -> 전체로 늘리는 것이 가장 큰 성능 향상 요인. dev 다수결로 추가 데이터 확보 |
| **Epoch** | 5 | 데이터 전체를 충분히 학습. A100이라 시간적 여유 있음 |
| **LR** | 2e-5 | 베이스라인 1e-4는 너무 높아서 불안정. 2e-5는 대형 모델 파인튜닝 표준값 |
| **LoRA** | r=32, alpha=64 | rank가 높을수록 표현력 증가. 7B 모델에는 r=32가 적절 |
| **IMAGE_SIZE** | 512 | 384 -> 512로 올리면 재활용품의 라벨, 텍스처 등 세밀한 특징 인식 향상 |
| **Labels 마스킹** | 적용 | 시스템/유저 프롬프트는 loss에서 제외 -> assistant 답변만 학습해서 효율 극대화 |
| **GRAD_ACCUM** | 8 | 실질 배치 8 -> 학습 안정성 향상 |
| **프롬프트** | 한국어 재활용 특화 | 도메인 특화 지시가 범용 영어보다 재활용품 인식에 유리 |

# 환경 준비
- 런타임 -> 런타임 유형 변경 -> GPU (A100)
- 아래 셀 실행 후 런타임 재시작

In [1]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade

  Installing build dependencies ... canceled
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 76, in resolve
    collected = self.factory.collect_root_requirements(root_reqs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/factory.py", line 

In [2]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name())

Torch version: 2.10.0+cpu
CUDA version: None


AssertionError: Torch not compiled with CUDA enabled

# 데이터 준비
구글 드라이브에 데이터 zip 파일을 업로드한 후 실행

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 파일 이름이 이미지와 동일한지 확인하세요!
!unzip -q "/content/260401_15_2_ai.zip" -d "/content/"

# 압축이 잘 풀렸는지 파일 목록 확인
!ls -l /content/

# 라이브러리, 데이터, 설정

In [ ]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from collections import Counter
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ===== 실험 설정 =====
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"   # 7B 모델 (3B 대비 이해력 향상)
IMAGE_SIZE = 512                              # 384 -> 512 (디테일 인식 향상)
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ===== 데이터 로드 =====
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

# [변경] 전체 데이터 사용 (베이스라인은 200개만 사용했음)
# train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)  # 주석 처리!

# ===== dev.csv pseudo-label 추가 =====
# dev.csv의 응답1~5를 다수결로 정답 추정 -> 학습 데이터에 합치기
dev_df = pd.read_csv("/content/dev.csv")
pseudo_labels = []
for _, row in dev_df.iterrows():
    responses = []
    for i in range(1, 6):
        col = f"응답{i}"
        if col in dev_df.columns and pd.notna(row[col]):
            r = str(row[col]).strip().lower()
            if r in ["a","b","c","d"]:
                responses.append(r)
    if responses:
        pseudo_labels.append(Counter(responses).most_common(1)[0][0])
    else:
        pseudo_labels.append("a")

dev_df["answer"] = pseudo_labels
dev_for_train = dev_df[["id","path","question","a","b","c","d","answer"]].copy()

# train + dev 합치기
train_df = pd.concat([train_df, dev_for_train], ignore_index=True)
print(f"총 학습 데이터: {len(train_df)}개 (train + dev pseudo-label)")
print(f"테스트 데이터: {len(test_df)}개")

# 모델, Processor

7B 모델 다운로드: 약 15GB, 10~20분 소요

In [ ]:
# 4bit 양자화 설정
# -> A100에서도 4bit를 쓰면 VRAM 여유가 생겨 해상도(512)와 배치를 올릴 수 있음
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 프로세서 (이미지 해상도 512x512로 설정)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE*IMAGE_SIZE,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
)

# 7B 사전학습 모델 로드
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA: r=32 (베이스라인 r=8 대비 표현력 4배 증가)
# -> 7B 모델은 파라미터가 많아서 높은 rank가 효과적
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 프롬프트 (한국어 재활용 특화)

**왜 한국어?** 재활용품 이미지의 라벨이 한국어이고, 질문도 한국어. 도메인 특화 지시가 범용 영어보다 유리.

In [ ]:
SYSTEM_INSTRUCT = (
    "당신은 재활용품 이미지를 분석하는 전문 AI입니다. "
    "이미지의 소재, 색상, 형태, 라벨을 주의 깊게 관찰하세요. "
    "질문에 대해 a, b, c, d 중 정답 하나만 소문자로 출력하세요."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        "이미지를 자세히 살펴보고 아래 질문에 답하세요.\n\n"
        f"질문: {question}\n\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답: "
    )

# Dataset, Collator (Labels 마스킹 적용)

**Labels 마스킹이란?** 베이스라인은 시스템 프롬프트 + 유저 질문 + 정답 전체에 loss를 계산함.  
-> 모델이 "프롬프트를 외우는 데" 학습 자원을 낭비.  
-> assistant 정답 부분만 loss 계산하면 학습 효율 대폭 향상.

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]
            text = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts, images=images, padding=True, return_tensors="pt"
        )

        if self.train:
            labels = enc["input_ids"].clone()
            # [핵심] assistant 응답 이전 토큰을 -100으로 마스킹
            # -> loss 계산에서 제외되어 정답 부분만 학습
            for i, text in enumerate(texts):
                assistant_marker = "assistant\n"
                marker_pos = text.rfind(assistant_marker)
                if marker_pos != -1:
                    prefix = text[:marker_pos + len(assistant_marker)]
                    prefix_ids = self.processor.tokenizer(
                        prefix, return_tensors="pt"
                    )["input_ids"].shape[1]
                    labels[i, :prefix_ids] = -100
            enc["labels"] = labels

        return enc

# DataLoader

In [ ]:
split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                          collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False,
                          collate_fn=DataCollator(processor, True), num_workers=0)

print(f"Train: {len(train_ds)}, Valid: {len(valid_ds)}")

# Fine-tuning

- 5 epoch, LR 2e-5, GRAD_ACCUM 8
- A100에서 전체 데이터 기준 약 1~2시간 예상

In [ ]:
model = model.to(device)
GRAD_ACCUM = 8          # 실질 배치 8 (베이스라인 4 대비 안정적)
EPOCHS = 5              # 5 epoch (베이스라인 1 대비 충분한 학습)
LR = 2e-5               # 2e-5 (베이스라인 1e-4 대비 안정적)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
num_training_steps = EPOCHS * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)
scaler = torch.amp.GradScaler('cuda', enabled=True)

best_val_loss = float('inf')
global_step = 0

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1
            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

    # Validation
    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    avg_val = val_loss/val_steps
    print(f"[Epoch {epoch+1}] valid loss {avg_val:.4f}")

    # Best 모델 저장 (Google Drive에 저장하여 세션 끊김 대비)
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        SAVE_DIR = "/content/drive/My Drive/best_model_colab_a100"
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"  -> Best model saved! (val_loss: {avg_val:.4f})")

print(f"\nTraining done. Best valid loss: {best_val_loss:.4f}")

# Inference

전체 test 데이터 추론. 5074개 기준 약 30분~1시간 소요.

In [ ]:
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last
    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=4, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    preds.append(extract_choice(output_text))

# 제출 파일 생성 (Google Drive에도 백업)
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission_colab_a100.csv", index=False)
submission.to_csv("/content/drive/My Drive/submission_colab_a100.csv", index=False)
print("Saved submission_colab_a100.csv")
print(submission.head(10))